<a href="https://colab.research.google.com/github/GitTanish/RAG/blob/master/RagAPP27.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import langchain
print(langchain.__version__)

1.2.15


In [ ]:
import os
os.environ['GROQ_API_KEY']=""

In [ ]:
os.environ["LANGCHAIN_TRACING"]="true"
os.environ["LANGCHAIN_API_KEY"]=""
os.environ["LANGCHAIN_PROJECT"]="langchain101"

In [ ]:
! pip install -qU langchain-groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")
llm_response = llm.invoke('Tell me a joke')
llm_response

AIMessage(content='Why don’t scientists trust atoms anymore?\n\nBecause they **make up** everything! 😄', additional_kwargs={'reasoning_content': 'User wants a joke. Simple. Provide a joke. Probably a short funny one.'}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 75, 'total_tokens': 120, 'completion_time': 0.093589388, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.003649589, 'prompt_tokens_details': None, 'queue_time': 0.004764108, 'total_time': 0.097238977}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9d1f936695', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d76df-5727-7253-83c2-3b4abf4ab70f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 45, 'total_tokens': 120, 'output_token_details': {'reasoning': 18}})

#### Parsing output


In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(llm_response)

'Why don’t scientists trust atoms anymore?\n\nBecause they **make up** everything! 😄'

#### Simple Chain

In [ ]:
chain = llm | output_parser
chain.invoke("How make a scrambled egg")

'Here’s a quick, fool‑proof way to make soft, fluffy scrambled eggs on the stovetop.  \nFeel free to adjust the quantities or add your favorite extras (cheese, herbs, veggies, etc.) to suit your taste.\n\n---\n\n## Ingredients (serves 1)\n\n| Ingredient | Amount |\n|------------|--------|\n| Large eggs | 2‑3 |\n| Milk, cream, or water | 1\u202f–\u202f2\u202fTbsp (optional, for extra fluffiness) |\n| Salt | pinch (to taste) |\n| Freshly ground black pepper | pinch (to taste) |\n| Butter or oil | 1\u202ftsp |\n| Optional add‑ins | shredded cheese, chopped herbs, diced ham, sautéed veggies, hot sauce, etc. |\n\n---\n\n## Equipment\n\n* Small non‑stick skillet or frying pan (about 8‑10\u202fin)\n* Silicone or rubber spatula\n* Small bowl\n* Fork or whisk\n* Plate\n\n---\n\n## Step‑by‑Step Method\n\n1. **Crack & Beat**  \n   * Crack the eggs into a bowl.  \n   * Add the milk/cream (or water) if you’re using it.  \n   * Sprinkle in a pinch of salt and pepper.  \n   * Beat vigorously with a f

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class WorkOutPlanner(BaseModel):
  routine: str = Field(description = "When does the speaker wakes and sleep")
  shoulder_exercise: str = Field(description="What are his shoulder specific exercise")
  diet: str = Field(description= "What food he eats for his diet")

transcription = """
Hey It's me XYZ, My day starts at waking up on 7 am , I start my day with scrambled eggs, milk and bread.
Then I work on my various project, at 3 pm. I hit the gym for workout. For chest I do bench press, I can bench a small house I feel.
For shoulders I do Shoulder press and Lateral Raises feels great tho. Like wise I divide the week different muscle groups.

I normally eat chicken breast with some butter , Mackerel and sometimes lobster.
I feel alive and great.

"""

structured_llm = llm.with_structured_output(WorkOutPlanner)
output = structured_llm.invoke(transcription)
output

In [ ]:
output.diet[0:14]

#### Prompt Template

In [ ]:
# dynamic prompting
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Best thing about {topic}, in 100 words")
prompt.invoke({"topic":"programming"})

ChatPromptValue(messages=[HumanMessage(content='Best thing about programming, in 100 words', additional_kwargs={}, response_metadata={})])

In [ ]:
chain = prompt | llm | output_parser
chain.invoke({"topic":"Flowerhorn"})

'Flowerhorns captivate with their vivid, metallic colors and the iconic nuchal hump that proudly declares their hybrid vigor. Their lively, curious nature makes them engaging companions, often recognizing owners and displaying playful antics. These fish thrive in well‑maintained, spacious aquariums where their striking patterns become living art, turning any tank into a dynamic centerpiece. The breed’s robust health and adaptability to varied water conditions simplify care, allowing both novice and experienced hobbyists to enjoy their presence. Ultimately, the Flowerhorn’s blend of beauty, personality, and resilience creates an unforgettable aquarium experience that delights every observer. Their charisma truly enriches any home.'

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# define the prompt
prompt = ChatPromptTemplate.from_template("Technical niche about {topic}")

# initialise the llm
llm = ChatGroq(model = "openai/gpt-oss-120b")

# Define the output parser
output_parser = StrOutputParser()

# compose the chain
chain = prompt | llm | output_parser

# use the chain
result = chain.invoke({"topic":"Cockatoo"})
print(result)

#### LLM Messages

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

system_message = SystemMessage(content= "You are an edgy internet tech influencer that does improv, you're job here is to make joke of a query")
human_message = HumanMessage(content = 'Tell me about programming')
llm.invoke([system_message, human_message])

In [ ]:
template = ChatPromptTemplate([
    ('system', "You're a struggling comedian that tells jokes."),
    ("human","tell me about {topic}")
])

prompt_value = template.invoke(
    {
        "topic":"Fish Curry"
    }
)

llm.invoke(prompt_value)

In [ ]:
prompt_value

In [ ]:
!pip install -U langchain langchain-community sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: sentence-transformers
    Found existing installation: senten

In [ ]:
!pip install docx2txt

  Using cached docx2txt-0.9-py3-none-any.whl.metadata (529 bytes)
Using cached docx2txt-0.9-py3-none-any.whl (4.0 kB)


In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import List
from langchain_core.documents import Document

# splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    length_function=len
)

# load docx
docx_loader = Docx2txtLoader("/content/docs/first_principles_bible.docx")
documents = docx_loader.load()

# split
splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks")



Split the documents into 99 chunks


In [ ]:
splits[0]

Document(metadata={'source': '/content/docs/first_principles_bible.docx'}, page_content='FIRST PRINCIPLES BIBLE   |   Built for Tanish Saroj\n\nFIRST PRINCIPLES\n\nB I B L E\n\n\n\nThinking from the Ground Up — for Engineers Who Build the Future\n\nCompiled for  TANISH SAROJ\n\nB.E. Computer Science | AI & LLMOps Engineer | SIH 2025 National Winner\n\nChapter 1: What First Principles Thinking Actually Is')

In [ ]:
# function to load documents from a folder
def load_documents(folder_path: str) -> List[Document]:
  documents = []
  for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if filename.endswith('.pdf'):
      loader = PyPDFLoader(file_path)
    elif filename.endswith('.docx'):
      loader = Docx2txtLoader(file_path)
    else:
      print(f"Unsupported file type: {filename}")
      continue
    documents.extend(loader.load())
  return documents

# load documents from a folder
folder_path = "/content/docs"
documents = load_documents(folder_path)

print(f"loaded {len(documents)} documents from the folder")
splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks")


loaded 4 documents from the folder
Split the documents into 102 chunks


In [ ]:
# 38:01

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# init model
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/e5-base-v2"
)

document_embeddings = embeddings.embed_documents(
    [f"passage: {split.page_content}" for split in splits]
)


print(f"Created embeddings for {len(document_embeddings)} documents chunks.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Created embeddings for 102 documents chunks.


In [ ]:
document_embeddings[0]

[0.00622699735686183,
 -0.03879019245505333,
 -0.06536301970481873,
 -0.016703393310308456,
 0.07073163986206055,
 -0.050269726663827896,
 0.046294666826725006,
 0.039149392396211624,
 -0.023749960586428642,
 -0.036265693604946136,
 -0.011757709085941315,
 0.0691920593380928,
 -0.04646370932459831,
 -0.028254469856619835,
 -0.03598723188042641,
 0.03025185689330101,
 0.027752500027418137,
 -0.03087444417178631,
 -0.008264507167041302,
 -0.01783514954149723,
 -0.04012234881520271,
 -0.006797562353312969,
 0.04978661984205246,
 0.0072967056185007095,
 0.0306243896484375,
 0.017903178930282593,
 0.01275046169757843,
 0.008358352817595005,
 -0.03252287581562996,
 -0.03248113766312599,
 0.028409138321876526,
 0.038549914956092834,
 0.02282419614493847,
 -0.05774381756782532,
 -0.04211458936333656,
 0.023744046688079834,
 -0.05632484331727028,
 0.019756928086280823,
 -0.0472414493560791,
 -0.011856608092784882,
 -0.023730965331196785,
 -0.03740944713354111,
 -0.03384353965520859,
 0.02580608

#### Create and persist Chroma vector store

In [ ]:
pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

embedding_function = HuggingFaceEmbeddings(
    model_name="intfloat/e5-base-v2"
)

collection_name = "my_collection"
vectorstore = Chroma.from_documents(collection_name=collection_name, documents= splits,
                                    embedding=embedding_function, persist_directory="./chroma_db")

print("Vector store created and persisted to './chroma_db'")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created and persisted to './chroma_db'


In [ ]:
# perform the similarity search

query = "what are the anti patterns"
search_results = vectorstore.similarity_search(query, k=2)

print(f"\nTop 2 most relevant chunks for the query: {query}\n")
for i, result in enumerate(search_results, 1):
  print(f"Result {i}:")
  print(f"Source: {result.metadata.get('source', 'unknown')}")
  print(f"Content: {result.page_content}")
  print()


Top 2 most relevant chunks for the query: Applications of first principles in engineering

Result 1:
Source: /content/docs/first_principles_bible.docx
Content: Chapter 2: How First Principles Has Been Applied — A Historical Map

Richard Feynman — The Notebook Method

Result 2:
Source: /content/docs/first_principles_bible.docx
Content: Chapter 7: Application in General Life and Day-to-Day Tasks

First principles thinking is not reserved for aerospace or AI. It applies to any situation where inherited assumptions might be leading you astray.

Career Decisions



In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2})
retriever.invoke("Habits to Specifically Avoid?")

[Document(id='8bdc0146-c179-4a8f-be34-01bb619b9322', metadata={'source': '/content/docs/first_principles_bible.docx'}, page_content='Habits to Specifically Avoid\n\n\n\nHABIT TO AVOID\n\nWHY IT DESTROYS FIRST PRINCIPLES THINKING\n\nCopying architecture from tutorials\n\nTutorials optimise for teachability, not for correctness. Their constraints are pedagogical, not real. Building production systems on tutorial architectures is building on someone else\'s assumptions.\n\n"This is industry standard"'),
 Document(id='2c43cb75-aa75-42a7-97a2-d51f17c8842e', metadata={'source': '/content/docs/first_principles_bible.docx'}, page_content='RULE\n\nIn engineering: never add a dependency you cannot explain at the level of what bytes it moves and what transforms it applies. If you cannot explain it, you cannot debug it.\n\nChapter 9: What to Avoid — The Anti-Patterns\n\nUnderstanding what to avoid is as important as understanding what to do. These are the failure modes that destroy first principle

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
template= """Answer the question based only on the following context:
{context}

Question: {question}

Answer: """
prompt = ChatPromptTemplate.from_template(template)


In [ ]:
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context":retriever, "question":RunnablePassthrough()} | prompt
)
rag_chain.invoke("Habits to Specifically Avoid?")

ChatPromptValue(messages=[HumanMessage(content='Answer the question based only on the following context:\n[Document(id=\'8bdc0146-c179-4a8f-be34-01bb619b9322\', metadata={\'source\': \'/content/docs/first_principles_bible.docx\'}, page_content=\'Habits to Specifically Avoid\\n\\n\\n\\nHABIT TO AVOID\\n\\nWHY IT DESTROYS FIRST PRINCIPLES THINKING\\n\\nCopying architecture from tutorials\\n\\nTutorials optimise for teachability, not for correctness. Their constraints are pedagogical, not real. Building production systems on tutorial architectures is building on someone else\\\'s assumptions.\\n\\n"This is industry standard"\'), Document(id=\'2c43cb75-aa75-42a7-97a2-d51f17c8842e\', metadata={\'source\': \'/content/docs/first_principles_bible.docx\'}, page_content=\'RULE\\n\\nIn engineering: never add a dependency you cannot explain at the level of what bytes it moves and what transforms it applies. If you cannot explain it, you cannot debug it.\\n\\nChapter 9: What to Avoid — The Anti-Pat

In [ ]:
def docs2str(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context":retriever | docs2str, "question":RunnablePassthrough()} | prompt
)
rag_chain.invoke("How Elon Musk Thinks — The SpaceX Decomposition?")

ChatPromptValue(messages=[HumanMessage(content='Answer the question based only on the following context:\nChapter 3: How Elon Musk Thinks — The SpaceX Decomposition\n\nElon Musk is the most widely cited practitioner of first principles thinking alive today. Understanding his methodology concretely is worth more than any abstract description.\n\nThe SpaceX Case Study\n\nIn 2002, Musk wanted to buy a rocket to send a greenhouse to Mars as a publicity stunt to reignite public interest in space. Russian rockets cost $65 million each. He walked away.\n\nHe did not conclude "rockets should be cheaper" and advocate for reform. He concluded: if we redesign the manufacturing process from first principles, we can build our own rockets at a fraction of the cost. SpaceX was founded.\n\nMusk\'s Explicit Algorithm\n\nQuestion the requirement. Every requirement has a source. If that source is a person (not physics), interrogate it. Requirements that come from someone\'s preference are not constraints

In [ ]:
rag_chain = (
    {"context":retriever| docs2str, "question":RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

question = "The Notebook Method?"
response = rag_chain.invoke(question)
print(response)

**The Notebook Method** – as attributed to Richard Feynman – is a disciplined habit of recording your thoughts, questions, and the step‑by‑step reasoning behind any belief, assumption, or decision you’re examining. By writing everything down in a notebook, you force yourself to:

1. **Externalize** the idea so you can see it clearly.  
2. **Break it into smaller parts**, asking “why?” at each stage (much like the Five‑Whys exercise).  
3. **Track the logical chain** from the initial statement to its underlying foundations, exposing whether it rests on solid, verifiable truth (bedrock) or on unexamined, inherited assumptions (sand).  

Feynman used this method to keep his thinking transparent, to spot hidden premises, and to develop a habit of first‑principles analysis. The notebook becomes a personal laboratory where ideas are tested, refined, and either confirmed or discarded.


#### Conversational RAG

- Handling follow Up Questions

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
chat_history = []
chat_history.extend([
    HumanMessage(content= question),
    AIMessage(content = response)
])


In [ ]:
chat_history

[HumanMessage(content='The Notebook Method?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='**The Notebook Method** – as attributed to Richard\u202fFeynman – is a disciplined habit of recording your thoughts, questions, and the step‑by‑step reasoning behind any belief, assumption, or decision you’re examining. By writing everything down in a notebook, you force yourself to:\n\n1. **Externalize** the idea so you can see it clearly.  \n2. **Break it into smaller parts**, asking “why?” at each stage (much like the Five‑Whys exercise).  \n3. **Track the logical chain** from the initial statement to its underlying foundations, exposing whether it rests on solid, verifiable truth (bedrock) or on unexamined, inherited assumptions (sand).  \n\nFeynman used this method to keep his thinking transparent, to spot hidden premises, and to develop a habit of first‑principles analysis. The notebook becomes a personal laboratory where ideas are tested, refined, and either confirmed o

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question"
    "which might reference context in the chat history,"
    "formulate a standalone question which can be understood"
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as it."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder('chat_history'),
        ("human","{input}"),

    ]
)

contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()
contextualize_chain.invoke({"input": "ok list the features of blockchain", "chat_history": chat_history})

'What are the key features of blockchain technology?'

In [ ]:
!pip install langchain-classic


In [ ]:
from langchain_classic.chains import create_history_aware_retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

history_aware_retriever.invoke({
    "input": "ok list the features of blockchain.",
    "chat_history": chat_history
})

[Document(id='3fce53e8-c19f-40e6-9edd-a2ed95ece552', metadata={'source': '/content/docs/rag_test_doc_3.docx'}, page_content='Blockchain Basics\n\nBlockchain is a distributed ledger technology that ensures secure and transparent transactions.\n\nFeatures:\n- Decentralization\n- Immutability\n- Transparency\n\nUse cases:\n- Cryptocurrencies\n- Supply chain tracking\n- Smart contracts\n\nLimitations include scalability and energy consumption.'),
 Document(id='5d0a9607-c004-49d8-9371-163fe42f511e', metadata={'source': '/content/docs/rag_test_doc_3.docx'}, page_content='Blockchain Basics\n\nBlockchain is a distributed ledger technology that ensures secure and transparent transactions.\n\nFeatures:\n- Decentralization\n- Immutability\n- Transparency\n\nUse cases:\n- Cryptocurrencies\n- Supply chain tracking\n- Smart contracts\n\nLimitations include scalability and energy consumption.')]

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages([
    ("system","You are an helpful AI assistant. Use the following context to answer the user's question."),
    ("system","Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

question_answer_chain= create_stuff_documents_chain(llm, qa_prompt)

rag_chain= create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [ ]:
rag_chain.invoke({"input":"State only one feature of blockchain", "chat_history":chat_history})

{'input': 'State only one feature of blockchain',
 'chat_history': [HumanMessage(content='Features of Blockchain?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The features of blockchain are:\n\n- Decentralization  \n- Immutability  \n- Transparency', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(id='3fce53e8-c19f-40e6-9edd-a2ed95ece552', metadata={'source': '/content/docs/rag_test_doc_3.docx'}, page_content='Blockchain Basics\n\nBlockchain is a distributed ledger technology that ensures secure and transparent transactions.\n\nFeatures:\n- Decentralization\n- Immutability\n- Transparency\n\nUse cases:\n- Cryptocurrencies\n- Supply chain tracking\n- Smart contracts\n\nLimitations include scalability and energy consumption.'),
  Document(id='780e1c10-40b4-4ecf-a312-2dd9fc7df9ee', metadata={'source': '/content/docs/rag_test_doc_1.docx'}, page_content='AI in Healthcare\n\nArtificial Intelligence (AI) is trans

#### Building Multi User Chatbot

In [ ]:
import sqlite3

DB_NAME = "rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def create_application_logs():
    conn = get_db_connection()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS application_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id TEXT,
            user_query TEXT,
            gpt_response TEXT,
            model TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.close()

def insert_application_logs(session_id, user_query, gpt_response, model):
    conn = get_db_connection()
    conn.execute(
        'INSERT INTO application_logs (session_id, user_query, gpt_response, model) VALUES (?,?,?,?)',
        (session_id, user_query, gpt_response, model)
    )
    conn.commit()
    conn.close()

def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        'SELECT user_query, gpt_response FROM application_logs WHERE session_id=? ORDER BY created_at',
        (session_id,)
    )

    messages = []
    for row in cursor.fetchall():
        messages.extend([
            {"role": "human", "content": row['user_query']},
            {"role": "ai", "content": row['gpt_response']}
        ])

    conn.close()
    return messages

create_application_logs

<function __main__.create_application_logs()>

In [ ]:
import sqlite3
import uuid

create_application_logs()

session_id = str(uuid.uuid4())

# first query
chat_history = get_chat_history(session_id)

question1 = "What are the challenges mentioned?"
answer1 = rag_chain.invoke({
    "input": question1,
    "chat_history": chat_history
})['answer']

insert_application_logs(session_id, question1, answer1, "gpt-oss-120b")

print(answer1)

# second query (memory test)
chat_history = get_chat_history(session_id)

question2 = "summarize them"
answer2 = rag_chain.invoke({
    "input": question2,
    "chat_history": chat_history
})['answer']

insert_application_logs(session_id, question2, answer2, "gpt-oss-120b")

print(answer2)

Below is a concise list of the **key challenges** that keep most people from applying the three‑core‑property workflow (Decomposition → Validation → Reconstruction) as described in the material you’re working with.

| # | Challenge (what stops you) | Why it matters | Typical symptom (what you’ll notice) |
|---|----------------------------|----------------|--------------------------------------|
| 1 | **Failure to de‑compose** – trying to solve a problem as a monolith | You never isolate the “irreducible components” that can be verified. | You feel overwhelmed, jump straight to a solution, or keep “spinning your wheels.” |
| 2 | **Assumption‑driven thinking** – accepting unverified premises | Violates the **Validation** rule; you build on shaky foundations. | You hear yourself say “I know X is true” without any data or experiment to back it up. |
| 3 | **Premature reconstruction** – assembling a solution before every piece is validated | Leads to rework, hidden bugs, or outright failure

In [ ]:
!pip install nbformat